# CETI Analysis Pipeline
### Credit and Liquidity Risk Indicator, Complete Estimation, Testing & Export
**Thesis:** Tail Risk in Peruvian D-SIB Banking (BBVA & BCP), 2007–2026  
**Supervisor:** Prof. Argimiro Arratia Quesada  

--
**Before running this notebook:**  
Make sure `df_master_export.csv` exists. If not, run this in your pipeline notebook first:
```python
df_master.to_csv('OUTPUT_CLEAN_PIPELINE/df_master_export.csv')
```
Then run all cells top to bottom (**Run All** or `Ctrl+Alt+R`).


## Section 0, Imports & Configuration


In [1]:
import os, sys, warnings
import numpy as np
import pandas as pd
import scipy.stats as sp_stats
import statsmodels.api as sm
import matplotlib
matplotlib.use('Agg')  # save figures without a display window
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection
import matplotlib.dates as mdates
from scipy.stats import gaussian_kde
warnings.filterwarnings('ignore')

# ── Paths ────────────────────────────────────────────────────────────────
BASE      = '.'  # repo root
DATA_CSV  = '0607version/CETI_PIPELINE/df_master_export.csv'
OUT_DIR   = '0607version/CETI_ANALYSIS'
PLOTS_DIR = os.path.join(OUT_DIR, 'plots')
TABS_DIR  = os.path.join(OUT_DIR, 'tables')
for d in [OUT_DIR, PLOTS_DIR, TABS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Column names (matched to df_master) ─────────────────────────────────
CDS_H      = 'delta_cds_h_M1'       # System credit spread ΔCDS^H (Phase 1, LOO)
CDS_L      = 'delta_cds_l'          # Liquidity spread
CDS_L1     = 'delta_cds_l_lag1'     # Lagged liquidity spread
SR         = 'short_rate'           # Short-term interest rate
TS         = 'term_spread'          # Term spread (10y minus 3m)
CPI        = 'cpi_12m_pct'          # 12-month CPI inflation
M_ORTH_LOG = 'delta_m_orth_log_M1'  # Orthogonalised market return (log, preferred)
M_ORTH_RAW = 'delta_m_orth_M1'      # Fallback: raw scale
CONTROLS   = [CDS_L, CDS_L1, SR, TS, CPI]
BANKS      = ['BBVA', 'BCP']
DP         = {'BBVA': 'dp_BBVA', 'BCP': 'dp_BCP'}
WINDOWS    = [52, 104]
W_PRI      = 104  # Primary window for reporting and LSTM feature

# ── Crisis episodes: (start, end, label, type) ──────────────────────────
# type 'cl' = credit/liquidity stress  |  'pr' = political risk
CRISES = [
    ('2008-09-01','2009-06-30','GFC',             'cl'),
    ('2011-07-01','2012-06-30','Euro Debt Crisis', 'cl'),
    ('2015-08-01','2016-03-31','China/EM Selloff', 'cl'),
    ('2020-02-01','2020-09-30','COVID-19',         'cl'),
    ('2021-04-01','2021-12-31','Peru Political',   'pr'),
    ('2022-01-01','2022-12-31','Rate Hike Cycle',  'cl'),
]
CL_CRISES = [c for c in CRISES if c[3]=='cl']

# ── Colours ──────────────────────────────────────────────────────────────
C = {
    'BBVA' :'#1f4e79', 'BCP':'#843c0c',
    'cl'   :'#ffd966', 'pr' :'#d6b4fc',
    'hvuln':'#c00000', 'calm':'#70ad47', 'anom':'#9e0079',
    'zero' :'#888888', 'sig' :'#ff9999',
}
REG_COLORS = {'high_vuln':C['hvuln'], 'calm':C['calm'], 'anomalous':C['anom']}
print('Configuration loaded.')


Configuration loaded.


## Section, -- Helper Functions (run this cell once)


In [2]:
# ── Utilities ────────────────────────────────────────────────────────────
def sep(title='', w=68):
    pad = max(0,(w-len(title)-2)//2)
    print('─'*pad+(f' {title} ' if title else '')+'─'*pad)

def sig_star(p):
    if p<0.01: return '***'
    if p<0.05: return '**'
    if p<0.10: return '*'
    return '(ns)'

def nw_lags(T): return max(1,int(np.floor(4*(T/100)**(2/9))))

def make_dummy(index, crises, cl_only=False):
    d = pd.Series(0, index=index, dtype=int)
    for s,e,lbl,typ in crises:
        if cl_only and typ!='cl': continue
        d[(index>=pd.Timestamp(s))&(index<=pd.Timestamp(e))]=1
    return d

def shade(ax, index=None, crises=None):
    if crises is None: crises=CRISES
    for s,e,lbl,typ in crises:
        ax.axvspan(pd.Timestamp(s),pd.Timestamp(e),alpha=0.22,color=C[typ],zorder=0)

def cliffs_d(x,y):
    x=np.asarray(x,float); x=x[~np.isnan(x)]
    y=np.asarray(y,float); y=y[~np.isnan(y)]
    if len(x)==0 or len(y)==0: return np.nan
    cnt=sum((1 if xi>yi else -1 if xi<yi else 0) for xi in x for yi in y)
    return cnt/(len(x)*len(y))

def effect_label(d):
    a=abs(d)
    if a>=0.33: return 'large'
    if a>=0.147: return 'medium'
    return 'small'

# ── Rolling WLS ──────────────────────────────────────────────────────────
def rolling_wls(df, dp_col, W, m_orth):
    """
    Rolling WLS estimator for beta_H_t (CLRI credit-loading coefficient).
    Model: Dp_t = a + b_H*dCDS_H + b_M*Morth + controls + e
    Returns DataFrame: bh, se, ts, pv, r2, n  (one row per week)
    """
    feat=[CDS_H, m_orth]+CONTROLS
    need=[dp_col,'_w']+feat
    dv=df[need].dropna()
    out=pd.DataFrame(np.nan,index=df.index,columns=['bh','se','ts','pv','r2','n'])
    idx=dv.index
    req=max(len(feat)+2, W//2)
    for i in range(W-1,len(idx)):
        win=dv.loc[idx[max(0,i-W+1):i+1]]
        if len(win)<req: continue
        y=win[dp_col].values
        X=sm.add_constant(win[feat].values,has_constant='add')
        w=win['_w'].values
        try:
            m=sm.WLS(y,X,weights=w).fit()
            b=float(m.params[1]); se=float(m.bse[1])
            t=float(m.tvalues[1]); p2=float(m.pvalues[1])
            out.loc[idx[i]]=[b,se,t,p2/2 if b<0 else 1-p2/2,
                             float(m.rsquared),int(m.nobs)]
        except: pass
    return out.infer_objects()

# ── Full-sample WLS + HAC ─────────────────────────────────────────────────
def fullsample_hac(df, dp_col, m_orth, crisis_dummy=None):
    """
    Full-sample WLS with Newey-West HAC standard errors.
    Optional: crisis_dummy adds interaction term (state-dependent model).
    Returns dict with model, table, key estimates.
    """
    dfw=df.copy(); feat=[CDS_H,m_orth]+CONTROLS
    if crisis_dummy is not None:
        dfw['_Dcr']=crisis_dummy
        dfw['_int']=dfw[CDS_H]*dfw['_Dcr']
        feat=[CDS_H,'_int',m_orth]+CONTROLS
    need=[dp_col,'_w']+feat
    dv=dfw[need].dropna()
    y=dv[dp_col].values; X=sm.add_constant(dv[feat].values,has_constant='add')
    T=len(y)
    m=sm.WLS(y,X,weights=dv['_w'].values).fit(
        cov_type='HAC',cov_kwds={'maxlags':nw_lags(T),'use_correction':True})
    bh=float(m.params[1]); ph=float(m.pvalues[1])/2 if bh<0 else 1-float(m.pvalues[1])/2
    names=['const','beta_H_base']+(['beta_H_crisis'] if crisis_dummy is not None else [])+['DeltaM_orth']+CONTROLS
    rows=[{'Variable':n,'Coef':float(m.params[j]),'HAC_SE':float(m.bse[j]),
           't':float(m.tvalues[j]),'p(2-tail)':float(m.pvalues[j]),
           'Sig':sig_star(float(m.pvalues[j]))} for j,n in enumerate(names)]
    return {'m':m,'table':pd.DataFrame(rows),'bh':bh,'ph':ph,
            'bh_cr':float(m.params[2]) if crisis_dummy is not None else None,
            'ph_cr':float(m.pvalues[2]) if crisis_dummy is not None else None,
            'total_cr':(float(m.params[1])+float(m.params[2])) if crisis_dummy is not None else None,
            'r2':float(m.rsquared),'adj_r2':float(m.rsquared_adj),'N':T,'nw':nw_lags(T)}

# ── Statistical tests ─────────────────────────────────────────────────────
def sign_test(bh):
    v=bh.dropna(); nn=int((v<0).sum()); n=len(v)
    try:    p=float(sp_stats.binomtest(nn,n,0.5,alternative='greater').pvalue)
    except: p=float(sp_stats.binom_test(nn,n,0.5,alternative='greater'))
    return {'nn':nn,'n':n,'prop':nn/n,'p':p}

def wilcoxon_test(bh):
    v=bh.dropna().values
    if len(v)<10: return {'stat':np.nan,'p':np.nan,'med':np.nan}
    st,p=sp_stats.wilcoxon(v,alternative='less')
    return {'stat':float(st),'p':float(p),'med':float(np.median(v))}

def mw_test(bh, dummy):
    df2=pd.DataFrame({'b':bh,'d':dummy}).dropna()
    bc=df2.loc[df2.d==0,'b'].values; bk=df2.loc[df2.d==1,'b'].values
    if len(bc)<5 or len(bk)<5:
        return {k:np.nan for k in ['U','p','d','eff','nc','nk','mc','mk','medc','medk']}
    U,p=sp_stats.mannwhitneyu(bk,bc,alternative='less')
    d=cliffs_d(bk,bc)
    return {'U':float(U),'p':float(p),'d':float(d),'eff':effect_label(d),
            'nc':len(bc),'nk':len(bk),'mc':float(np.mean(bc)),'mk':float(np.mean(bk)),
            'medc':float(np.median(bc)),'medk':float(np.median(bk))}

def andrews_supwald(df, dp_col, m_orth, trim=0.15):
    feat=[CDS_H,m_orth]+CONTROLS; need=[dp_col,'_w']+feat
    dv=df[need].dropna()
    y=dv[dp_col].values; X=sm.add_constant(dv[feat].values,has_constant='add')
    T,k=len(y),X.shape[1]; st,en=int(trim*T),int((1-trim)*T)
    rss_r=float(sm.OLS(y,X).fit().ssr)
    ws=np.full(T,np.nan)
    for bp in range(st,en):
        try:
            r1=float(sm.OLS(y[:bp],X[:bp]).fit().ssr)
            r2=float(sm.OLS(y[bp:],X[bp:]).fit().ssr)
            denom=(r1+r2)/max(T-2*k,1)
            if denom>0: ws[bp]=((rss_r-r1-r2)/k)/denom
        except: pass
    sw=float(np.nanmax(ws)); bpi=int(np.nanargmax(ws))
    bpd=dv.index[bpi] if bpi<len(dv.index) else None
    pap='<0.01' if sw>=12.16 else '<0.05' if sw>=8.85 else '<0.10' if sw>=6.02 else '>0.10'
    return {'sw':sw,'bpd':bpd,'pap':pap,'ws':ws,'idx':dv.index}

def get_regimes(bh):
    v=bh.dropna()
    nt=float(v.mean()-v.std()); pt=float(v.mean()+v.std())
    reg=pd.Series('calm',index=bh.index,dtype=object)
    reg[bh<nt]='high_vuln'; reg[bh>pt]='anomalous'; reg[bh.isna()]=np.nan
    return reg,nt,pt

print('All helper functions defined.')


All helper functions defined.


## Section 1, Data Loading & Validation
Reads `df_master_export.csv`, selects the correct market-return column, and adds uniform WLS weights.


In [3]:
sep('Section 1, Data Loading')
if not os.path.exists(DATA_CSV):
    raise FileNotFoundError(
        f'Cannot find {DATA_CSV}.\n'
        'Run this in your pipeline notebook first:\n'
        '  df_master.to_csv("<path>/OUTPUT_CLEAN_PIPELINE/df_master_export.csv")')

df = pd.read_csv(DATA_CSV, index_col=0, parse_dates=True)
df.index = pd.to_datetime(df.index)
print(f'Data loaded:  {df.index.min().date()} → {df.index.max().date()}  ({len(df)} weeks)')

# Select market-return column (log-return preferred)
M_ORTH = M_ORTH_LOG if M_ORTH_LOG in df.columns else M_ORTH_RAW
print(f'Market return column: {M_ORTH}')

# Add uniform weight column (OLS equivalent at bank-level)
df['_w'] = 1.0

# Build crisis dummies
D_cl  = make_dummy(df.index, CRISES, cl_only=True)   # credit/liquidity only (for H2)
D_all = make_dummy(df.index, CRISES, cl_only=False)  # all episodes (robustness)

# Validate columns
missing = [c for c in list(DP.values())+[CDS_H,M_ORTH]+CONTROLS if c not in df.columns]
if missing: print(f'WARNING, missing columns: {missing}')
else:        print(f'All required columns present. Ready.')


───────────────────── Section 1, Data Loading ─────────────────────
Data loaded:  2007-01-12 → 2026-05-08  (1009 weeks)
Market return column: delta_m_orth_log_M1
All required columns present. Ready.


## Section 2 -- Rolling CETI Estimation
Estimates β̂ᴴ_t by OLS in each rolling window of W = 52 and 104 weeks.  
**β̂ᴴ_t < 0** means bank equity falls when system credit spreads widen: the core CETI signal.


In [4]:
sep('Section 2, Rolling Estimation (W=52 and W=104)')
RR = {}   # RR[bank][W] = DataFrame(bh, se, ts, pv, r2, n)

for bank in BANKS:
    RR[bank] = {}
    for W in WINDOWS:
        print(f'  {bank} W={W}...', end=' ', flush=True)
        rr = rolling_wls(df, DP[bank], W, M_ORTH)
        RR[bank][W] = rr
        bh = rr['bh'].dropna()
        print(f'N={len(bh)}  beta_H<0: {100*(bh<0).mean():.1f}%  mean={bh.mean():.5f}')

print()
print(f'  {"Bank":<6} {"W":>4} {"N":>5} {"Mean β̂ᴴ":>12} {"Std":>10} {"% <0":>7} {"% t<-1.96":>10}')
print('  '+'-'*55)
for bank in BANKS:
    for W in WINDOWS:
        bh=RR[bank][W]['bh'].dropna(); ts=RR[bank][W]['ts'].dropna()
        print(f'  {bank:<6} {W:>4} {len(bh):>5} {bh.mean():>12.5f} {bh.std():>10.5f}'
              f' {100*(bh<0).mean():>7.1f}% {100*(ts<-1.96).mean():>10.1f}%')


────────── Section 2, Rolling Estimation (W=52 and W=104) ──────────
  BBVA W=52... N=671  beta_H<0: 59.0%  mean=-0.00117
  BBVA W=104... N=619  beta_H<0: 75.3%  mean=-0.00285
  BCP W=52... N=670  beta_H<0: 47.8%  mean=0.00049
  BCP W=104... N=618  beta_H<0: 57.0%  mean=-0.00217

  Bank      W     N     Mean β̂ᴴ        Std    % <0  % t<-1.96
  -------------------------------------------------------
  BBVA     52   671     -0.00117    0.01065    59.0%       13.1%
  BBVA    104   619     -0.00285    0.00471    75.3%        8.1%
  BCP      52   670      0.00049    0.01655    47.8%        6.9%
  BCP     104   618     -0.00217    0.00870    57.0%        3.1%


## Section 3, Full-Sample WLS with Newey-West HAC Standard Errors
**Why HAC?** Weekly equity returns inherit serial correlation from slowly-moving controls (CPI, term spread). Newey-West corrects for this, giving reliable inference.  
Bandwidth: Andrews (1991) automatic selection = floor(4·(T/100)^(2/9)) ≈ 6 lags.


In [5]:
sep('Section 3, Full-Sample WLS (Newey-West HAC)')
FS = {}
for bank in BANKS:
    print(f'\n── {bank} ──────────────────────────────────')
    res = fullsample_hac(df, DP[bank], M_ORTH)
    FS[bank] = res
    t = res['table']
    print(f'  {"Variable":<30} {"Coef":>10} {"HAC_SE":>10} {"t":>7} {"p(2-tail)":>10} {"Sig":>6}')
    print('  '+'-'*75)
    for _, r in t.iterrows():
        print(f'  {r["Variable"]:<30} {r["Coef"]:>10.5f} {r["HAC_SE"]:>10.5f}'
              f' {r["t"]:>7.3f} {r["p(2-tail)"]:>10.4f} {r["Sig"]:>6}')
    print(f'  R²={res["r2"]:.4f}  Adj.R²={res["adj_r2"]:.4f}  N={res["N"]}  NW lags={res["nw"]}')
    print(f'  >>> beta_H = {res["bh"]:+.5f}   one-sided p (H1: beta_H<0) = {res["ph"]:.4f} {sig_star(res["ph"])}')


─────────── Section 3, Full-Sample WLS (Newey-West HAC) ───────────

── BBVA ──────────────────────────────────
  Variable                             Coef     HAC_SE       t  p(2-tail)    Sig
  ---------------------------------------------------------------------------
  const                             0.00079    0.00101   0.786     0.4319   (ns)
  beta_H_base                      -0.00224    0.00155  -1.442     0.1492   (ns)
  DeltaM_orth                       0.19648    0.01356  14.492     0.0000    ***
  delta_cds_l                      -0.01541    0.00393  -3.920     0.0001    ***
  delta_cds_l_lag1                 -0.00651    0.00232  -2.812     0.0049    ***
  short_rate                       -0.00001    0.00023  -0.040     0.9677   (ns)
  term_spread                      -0.00068    0.00035  -1.945     0.0518      *
  cpi_12m_pct                      -0.00013    0.00018  -0.722     0.4705   (ns)
  R²=0.3516  Adj.R²=0.3453  N=722  NW lags=6
  >>> beta_H = -0.00224   one-sided 

## Section 4, Rolling Significance Analysis
Proportion of rolling windows where β̂ᴴ is negative and/or significant.  
Complements the full-sample estimate: even if the average is small, a high fraction of negative windows signals a persistent directional tendency.


In [6]:
sep('Section 4, Rolling Significance (Level 2)')
for bank in BANKS:
    print(f'\n{bank}:')
    for W in WINDOWS:
        bh=RR[bank][W]['bh'].dropna(); ts=RR[bank][W]['ts'].dropna()
        print(f'  W={W:>3d}  N={len(bh)}  beta_H<0: {100*(bh<0).mean():.1f}%  '
              f't<-1.96: {100*(ts<-1.96).mean():.1f}%  '
              f't<-1.645: {100*(ts<-1.645).mean():.1f}%')


──────────── Section 4, Rolling Significance (Level 2) ────────────

BBVA:
  W= 52  N=671  beta_H<0: 59.0%  t<-1.96: 13.1%  t<-1.645: 22.2%
  W=104  N=619  beta_H<0: 75.3%  t<-1.96: 8.1%  t<-1.645: 8.7%

BCP:
  W= 52  N=670  beta_H<0: 47.8%  t<-1.96: 6.9%  t<-1.645: 10.3%
  W=104  N=618  beta_H<0: 57.0%  t<-1.96: 3.1%  t<-1.645: 10.8%


## Section 5, H1 Distributional Tests: Sign Test + Wilcoxon
**H₀:** β̂ᴴ is centered at zero.  **H₁:** β̂ᴴ < 0 in distribution.

- **Sign test:** Is the fraction of negative windows > 50% (binomial)?  
- **Wilcoxon signed-rank:** Is the median β̂ᴴ significantly below zero?  

*Note: overlapping windows create serial correlation → p-values are conservative.*


In [7]:
sep('Section 5, H1 Distributional Tests (Level 3)')
SR_res={}; WR_res={}
for bank in BANKS:
    SR_res[bank]={}; WR_res[bank]={}
    print(f'\n{bank}:')
    for W in WINDOWS:
        bh=RR[bank][W]['bh']
        s=sign_test(bh);     SR_res[bank][W]=s
        w=wilcoxon_test(bh); WR_res[bank][W]=w
        print(f'  W={W:>3d}  Sign: {s["nn"]}/{s["n"]} negative ({100*s["prop"]:.1f}%) '
              f'p={s["p"]:.4f}{sig_star(s["p"])}   '
              f'Wilcoxon: median={w["med"]:+.6f} p={w["p"]:.4f}{sig_star(w["p"])}')


─────────── Section 5, H1 Distributional Tests (Level 3) ───────────

BBVA:
  W= 52  Sign: 396/671 negative (59.0%) p=0.0000***   Wilcoxon: median=-0.001008 p=0.0000***
  W=104  Sign: 466/619 negative (75.3%) p=0.0000***   Wilcoxon: median=-0.001203 p=0.0000***

BCP:
  W= 52  Sign: 320/670 negative (47.8%) p=0.8845(ns)   Wilcoxon: median=+0.001036 p=0.8199(ns)
  W=104  Sign: 352/618 negative (57.0%) p=0.0003***   Wilcoxon: median=-0.002222 p=0.0000***


## Section 6, H2 Regime Tests: Mann-Whitney U + Cliff's δ
**H₀:** β̂ᴴ has the same distribution in calm and crisis periods.  
**H₁:** β̂ᴴ is more negative (smaller) during credit/liquidity stress.  

- **Primary test:** credit/liquidity episodes only (excludes 2021 political shock)  
- **Robustness:** all episodes including political risk  

**Cliff's δ interpretation:** |δ| < 0.147 = small, 0.147–0.33 = medium, > 0.33 = large.


In [8]:
sep('Section 6, H2 Regime Tests (Level 4)')
MW_res={}
for bank in BANKS:
    MW_res[bank]={}
    bh=RR[bank][W_PRI]['bh']
    print(f'\n{bank} (W={W_PRI}):')
    for label,dummy in [('Credit/liquidity only',D_cl),('All crises (incl. political)',D_all)]:
        r=mw_test(bh,dummy); MW_res[bank][label]=r
        print(f'  [{label}]')
        print(f'    n_calm={r["nc"]}  n_crisis={r["nk"]}  U={r["U"]:.0f}  '
              f'p={r["p"]:.4f}{sig_star(r["p"])}  Cliff delta={r["d"]:+.3f} ({r["eff"]})')
        print(f'    mean_calm={r["mc"]:+.5f}  mean_crisis={r["mk"]:+.5f}')


─────────────── Section 6, H2 Regime Tests (Level 4) ───────────────

BBVA (W=104):
  [Credit/liquidity only]
    n_calm=499  n_crisis=120  U=23760  p=0.0002***  Cliff delta=-0.206 (medium)
    mean_calm=-0.00230  mean_crisis=-0.00514
  [All crises (incl. political)]
    n_calm=459  n_crisis=160  U=26955  p=0.0000***  Cliff delta=-0.266 (medium)
    mean_calm=-0.00228  mean_crisis=-0.00451

BCP (W=104):
  [Credit/liquidity only]
    n_calm=498  n_crisis=120  U=13851  p=0.0000***  Cliff delta=-0.536 (large)
    mean_calm=-0.00062  mean_crisis=-0.00861
  [All crises (incl. political)]
    n_calm=458  n_crisis=160  U=15162  p=0.0000***  Cliff delta=-0.586 (large)
    mean_calm=-0.00011  mean_crisis=-0.00807


## Section 7, Andrews (1993) Sup-Wald Structural Break Test
Tests H₀: β̂ᴴ is constant over the full sample vs. H₁: single structural break.  
Searches all candidate breakpoints (trimming 15% from each end) and reports the supremum Wald statistic.  

**Critical values** (Andrews 1993, Table 1, p=1): 10% = 6.02, 5% = 8.85, 1% = 12.16  

*Low sup-Wald does not contradict H2*, it means there is no single sharp break; instead β̂ᴴ switches regimes multiple times (which is consistent with the Mann-Whitney findings).


In [9]:
sep('Section 7, Andrews (1993) Sup-Wald Test')
AW={}
for bank in BANKS:
    res=andrews_supwald(df,DP[bank],M_ORTH)
    AW[bank]=res
    bpd=str(res['bpd'].date()) if res['bpd'] else 'N/A'
    print(f'{bank}: sup-Wald={res["sw"]:.2f}  (CV 5%=8.85)  p≈{res["pap"]}  '
          f'Optimal break: {bpd}')
    verdict = 'H0 REJECTED, parameter instability confirmed' if '<' in res['pap'] \
              else 'H0 not rejected (no single dominant break; regime-switching pattern)'
    print(f'  → {verdict}')


───────────── Section 7, Andrews (1993) Sup-Wald Test ─────────────
BBVA: sup-Wald=2.23  (CV 5%=8.85)  p≈>0.10  Optimal break: 2020-05-08
  → H0 not rejected (no single dominant break; regime-switching pattern)
BCP: sup-Wald=2.85  (CV 5%=8.85)  p≈>0.10  Optimal break: 2021-07-02
  → H0 not rejected (no single dominant break; regime-switching pattern)


## Section 8, State-Dependent Model (Crisis Interaction)
Adds a crisis interaction term: β̂ᴴ_crisis = additional credit loading during stress.  

Model: Δp = α + β̂ᴴ_base·ΔCDS^H + β̂ᴴ_crisis·(D_crisis × ΔCDS^H) + βᴹ·Morth + controls + ε  

**Note:** If β̂ᴴ_crisis is insignificant or positive, this does not invalidate H1. It means the slope interaction is not detectable in this coarse-dummy specification, the rolling Mann-Whitney approach (Section 6) is more powerful for detecting regime differences.


In [10]:
sep('Section 8, State-Dependent Model')
SD={}
for bank in BANKS:
    print(f'\n── {bank} ──────────────────────────────────')
    res=fullsample_hac(df,DP[bank],M_ORTH,crisis_dummy=D_cl)
    SD[bank]=res
    for _,r in res['table'].iterrows():
        print(f'  {r["Variable"]:<30} {r["Coef"]:>10.5f} {r["HAC_SE"]:>10.5f}'
              f' {r["t"]:>7.3f} {r["p(2-tail)"]:>10.4f} {r["Sig"]:>6}')
    print(f'  R²={res["r2"]:.4f}  N={res["N"]}')
    print(f'  beta_H_base={res["bh"]:+.5f}  beta_H_crisis={res["bh_cr"]:+.5f}  '
          f'Total crisis={res["total_cr"]:+.5f}')


───────────────── Section 8, State-Dependent Model ─────────────────

── BBVA ──────────────────────────────────
  const                             0.00083    0.00102   0.816     0.4144   (ns)
  beta_H_base                      -0.00238    0.00155  -1.531     0.1259   (ns)
  beta_H_crisis                     0.00105    0.00284   0.371     0.7109   (ns)
  DeltaM_orth                       0.19610    0.01374  14.274     0.0000    ***
  delta_cds_l                      -0.01523    0.00411  -3.706     0.0002    ***
  delta_cds_l_lag1                 -0.00660    0.00228  -2.894     0.0038    ***
  short_rate                       -0.00002    0.00023  -0.066     0.9475   (ns)
  term_spread                      -0.00069    0.00035  -1.974     0.0483     **
  cpi_12m_pct                      -0.00014    0.00019  -0.752     0.4523   (ns)
  R²=0.3518  N=722
  beta_H_base=-0.00238  beta_H_crisis=+0.00105  Total crisis=-0.00132

── BCP ──────────────────────────────────
  const                   

## Section 9, Three-State Regime Classification
Each week is assigned to one of three states based on β̂ᴴ_W104:  

| State | Condition | Interpretation |
|--|--|--|
| **high_vuln** | β̂ᴴ < μ − 1σ | Strong negative loading, equity amplifies credit stress |
| **calm** | μ − 1σ ≤ β̂ᴴ ≤ μ + 1σ | Decoupled from credit conditions |
| **anomalous** | β̂ᴴ > μ + 1σ | Transmission broken/reversed (e.g., 2021 Peru political shock) |


In [11]:
sep('Section 9, Three-State Regime Classification')
REG={}; THR={}
for bank in BANKS:
    bh=RR[bank][W_PRI]['bh']
    reg,nt,pt=get_regimes(bh)
    REG[bank]=reg; THR[bank]=(nt,pt)
    counts=reg.dropna().value_counts(); tot=counts.sum()
    print(f'\n{bank}  (thresholds: neg={nt:.5f}, pos={pt:.5f})')
    for st in ['high_vuln','calm','anomalous']:
        n=counts.get(st,0)
        print(f'  {st:<15}: {n:>4} weeks ({100*n/tot:.1f}%)')


─────────── Section 9, Three-State Regime Classification ───────────

BBVA  (thresholds: neg=-0.00756, pos=0.00186)
  high_vuln      :   91 weeks (14.7%)
  calm           :  504 weeks (81.4%)
  anomalous      :   24 weeks (3.9%)

BCP  (thresholds: neg=-0.01087, pos=0.00653)
  high_vuln      :  113 weeks (18.3%)
  calm           :  404 weeks (65.4%)
  anomalous      :  101 weeks (16.3%)


## Section 10, LSTM Feature Export
**Primary LSTM feature:** `clri_BBVA_W104` and `clri_BCP_W104` (rolling β̂ᴴ_t).  

**Why the sensitivity (β̂ᴴ_t), not the contribution (β̂ᴴ_t × ΔCDS^H_t)?**  
Block 3 validation showed that the lagged contribution does NOT predict tail events. The sensitivity β̂ᴴ_t is a *regime-conditioning variable*: it tells the LSTM how exposed the bank currently is to credit conditions, which affects the *scale* of the tail distribution, not which specific week the crash occurs.  

**No look-ahead bias:** β̂ᴴ_t uses only data from [t-W, t], so it is legitimately available at time t for forecasting week t+1 VaR/ES.


In [12]:
sep('Section 10, LSTM Feature Export')
lstm=pd.DataFrame(index=df.index)
for bank in BANKS:
    for W in WINDOWS:
        lstm[f'clri_{bank}_W{W}']=RR[bank][W]['bh']
    lstm[f'regime_{bank}']=REG[bank]
lstm['dp_BBVA']=df.get('dp_BBVA'); lstm['dp_BCP']=df.get('dp_BCP')
lstm['dCDS_H']=df.get(CDS_H); lstm['D_crisis_cl']=D_cl

fp=os.path.join(OUT_DIR,'clri_lstm_final.csv')
lstm.to_csv(fp)
print(f'LSTM feature file saved: {fp}')
print(f'Primary LSTM inputs: clri_BBVA_W{W_PRI} / clri_BCP_W{W_PRI}')
print()
num_cols=[c for c in lstm.columns if pd.api.types.is_numeric_dtype(lstm[c])]
print(f'  {"Column":<40} {"N":>5} {"Mean":>10} {"Std":>10}')
print('  '+'-'*68)
for c in lstm.columns:
    s=lstm[c].dropna()
    if pd.api.types.is_numeric_dtype(s):
        print(f'  {c:<40} {len(s):>5} {s.mean():>+10.5f} {s.std():>10.5f}')


───────────────── Section 10, LSTM Feature Export ─────────────────
LSTM feature file saved: /Users/167yiliqi/Downloads/TEST - PER/OUTPUT_CLEAN_PIPELINE/clri_lstm_final.csv
Primary LSTM inputs: clri_BBVA_W104 / clri_BCP_W104

  Column                                       N       Mean        Std
  --------------------------------------------------------------------
  clri_BBVA_W52                              671   -0.00117    0.01065
  clri_BBVA_W104                             619   -0.00285    0.00471
  clri_BCP_W52                               670   +0.00049    0.01655
  clri_BCP_W104                              618   -0.00217    0.00870
  dp_BBVA                                   1007   +0.00046    0.01016
  dp_BCP                                    1001   +0.00156    0.02161
  dCDS_H                                     722   -0.00570    0.32167
  D_crisis_cl                               1009   +0.21407    0.41038


## Section 11, Figures (9 plots saved to PLOTS folder)
All figures are saved as PNG files (150 dpi) to `OUTPUT_CLEAN_PIPELINE/PLOTS/`.  

| # | File | Content |
|--|--|--|
| 1 | `01_rolling_sensitivity_W104.png` | Rolling β̂ᴴ (W=104), three-state coloured |
| 2 | `02_rolling_sensitivity_W52.png` | Rolling β̂ᴴ (W=52), appendix sensitivity check |
| 3 | `03_regime_timeline.png` | Three-state regime classification over time |
| 4 | `04_tstat_series.png` | Rolling t-statistics with significance thresholds |
| 5 | `05_regime_density_KDE.png` | KDE: calm vs crisis β̂ᴴ distributions |
| 6 | `06_andrews_supwald.png` | Sup-Wald statistic series with critical values |
| 7 | `07_statedependent_coefs.png` | State-dependent model coefficient plot |
| 8 | `08_significance_summary.png` | Four-level significance framework |
| 9 | `09_lstm_feature.png` | LSTM input feature (β̂ᴴ_W104, regime-coloured) |


In [13]:
sep('Section 11, Generating Figures')
plt.rcParams.update({'font.size':13,'axes.titlesize':14,'axes.labelsize':13,'legend.fontsize':11,'xtick.labelsize':11,'ytick.labelsize':11,'figure.dpi':150})

# ── Fig 1: Rolling sensitivity W=104 (main thesis figure) ───────────────
print('Figure 1: Rolling sensitivity W=104...')
fig,axes=plt.subplots(1,2,figsize=(18,7))
for ax,bank in zip(axes,BANKS):
    r=RR[bank][W_PRI]; bh=r['bh']; se=r['se']; reg=REG[bank]
    nt,pt=THR[bank]; idx=bh.dropna().index
    bank_col=C['BBVA'] if bank=='BBVA' else C['BCP']
    shade(ax)
    xnum=mdates.date2num(idx.to_pydatetime()); yval=bh.loc[idx].values
    pts=np.array([xnum,yval]).T.reshape(-1,1,2)
    segs=np.concatenate([pts[:-1],pts[1:]],axis=1)
    cols=[C['hvuln'] if v<nt else C['anom'] if v>pt else bank_col for v in yval]
    lc=LineCollection(segs,colors=cols,linewidths=1.4,zorder=3)
    ax.add_collection(lc)
    ax.set_xlim(xnum[0],xnum[-1]); ax.xaxis_date(); fig.autofmt_xdate()
    ax.fill_between(idx,(bh-1.96*se).loc[idx],(bh+1.96*se).loc[idx],alpha=0.12,color=bank_col)
    ax.axhline(0,color=C['zero'],ls='--',lw=0.8)
    ax.axhline(FS[bank]['bh'],color='#c55a11',ls=':',lw=1.2,label=f'Full-sample={FS[bank]["bh"]:.4f}')
    ax.axhline(nt,color=C['hvuln'],ls='--',lw=0.8,alpha=0.6)
    ax.axhline(pt,color=C['anom'], ls='--',lw=0.8,alpha=0.6)
    ax.legend(fontsize=11,loc='lower left')
    ax.set_ylabel(r'$\hat{\beta}^H_t$',fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR,'01_rolling_sensitivity_W104.png'),dpi=150,bbox_inches='tight')
plt.close(); print('  Saved.')

# ── Fig 2: W=52 (appendix) ───────────────────────────────────────────────
print('Figure 2: Rolling sensitivity W=52...')
fig,axes=plt.subplots(1,2,figsize=(18,6))
for ax,bank in zip(axes,BANKS):
    r=RR[bank][52]; bh=r['bh']; se=r['se']
    bank_col=C['BBVA'] if bank=='BBVA' else C['BCP']
    idx=bh.dropna().index; shade(ax)
    ax.plot(idx,bh.loc[idx],color=bank_col,lw=1.2)
    ax.fill_between(idx,(bh-1.96*se).loc[idx],(bh+1.96*se).loc[idx],alpha=0.12,color=bank_col)
    ax.axhline(0,color=C['zero'],ls='--',lw=0.8)
    ax.set_ylabel(r'$\hat{\beta}^H_t$',fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR,'02_rolling_sensitivity_W52.png'),dpi=150,bbox_inches='tight')
plt.close(); print('  Saved.')

# ── Fig 3: Regime timeline ────────────────────────────────────────────────
print('Figure 3: Three-state regime timeline...')
fig,axes=plt.subplots(2,1,figsize=(16,10))
for ax,bank in zip(axes,BANKS):
    bh=RR[bank][W_PRI]['bh']; reg=REG[bank]; nt,pt=THR[bank]
    bank_col=C['BBVA'] if bank=='BBVA' else C['BCP']
    idx=bh.dropna().index; prev=seg=None
    for t in idx:
        st=reg.loc[t]
        if st!=prev:
            if prev and seg: ax.axvspan(seg,t,alpha=0.22,color=REG_COLORS.get(str(prev),'white'))
            seg=t; prev=st
    if prev and seg: ax.axvspan(seg,idx[-1],alpha=0.22,color=REG_COLORS.get(str(prev),'white'))
    ax.plot(idx,bh.loc[idx],color=bank_col,lw=1.2,zorder=5)
    ax.axhline(0,color=C['zero'],ls='--',lw=0.8)
    ax.axhline(nt,color=C['hvuln'],ls=':',lw=0.9,alpha=0.7)
    ax.axhline(pt,color=C['anom'], ls=':',lw=0.9,alpha=0.7)
    counts=reg.dropna().value_counts(); tot=counts.sum()
    patches=[mpatches.Patch(color=REG_COLORS[s],alpha=0.6,
             label=f'{s}({counts.get(s,0)}w,{100*counts.get(s,0)/tot:.0f}%)')
             for s in ['high_vuln','calm','anomalous']]
    ax.legend(handles=patches,fontsize=11,loc='lower left')
    ax.set_ylabel(r'$\hat{\beta}^H_t$',fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR,'03_regime_timeline.png'),dpi=150,bbox_inches='tight')
plt.close(); print('  Saved.')

# ── Fig 4: Rolling t-stats ────────────────────────────────────────────────
print('Figure 4: Rolling t-statistics...')
fig,axes=plt.subplots(1,2,figsize=(18,6))
for ax,bank in zip(axes,BANKS):
    ts=RR[bank][W_PRI]['ts'].dropna()
    bank_col=C['BBVA'] if bank=='BBVA' else C['BCP']
    shade(ax)
    ax.plot(ts.index,ts.values,color=bank_col,lw=0.9,alpha=0.8)
    ax.fill_between(ts.index,ts.values,-1.96,where=(ts.values<-1.96),color=C['sig'],alpha=0.5)
    ax.axhline(-1.96,color='red',ls='--',lw=1.2,label='-1.96 (5%)')
    ax.axhline(-1.645,color='orange',ls=':',lw=1.0,label='-1.645 (10%)')
    ax.axhline(0,color=C['zero'],ls='--',lw=0.7)
    ax.legend(fontsize=11)
    ax.set_ylabel('t-value',fontsize=13)
plt.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR,'04_tstat_series.png'),dpi=150,bbox_inches='tight')
plt.close(); print('  Saved.')

# ── Fig 5: KDE density ────────────────────────────────────────────────────
print('Figure 5: Regime density KDE...')
fig,axes=plt.subplots(1,2,figsize=(14,6))
for ax,bank in zip(axes,BANKS):
    bh=RR[bank][W_PRI]['bh']
    df2=pd.DataFrame({'b':bh,'d':D_cl}).dropna()
    bc=df2.loc[df2.d==0,'b'].values; bk=df2.loc[df2.d==1,'b'].values
    for vals,lbl,col in [(bc,'Calm',C['calm']),(bk,'Crisis',C['hvuln'])]:
        if len(vals)>5:
            kde=gaussian_kde(vals,bw_method='silverman')
            xr=np.linspace(vals.min()*1.5,vals.max()*1.5,300)
            ax.plot(xr,kde(xr),color=col,lw=2,label=lbl)
            ax.axvline(vals.mean(),color=col,ls=':',lw=1.2)
            ax.fill_between(xr,kde(xr),alpha=0.1,color=col)
    mwr=MW_res[bank]['Credit/liquidity only']
    ax.axvline(0,color=C['zero'],ls='--',lw=0.8)
    ax.set_xlabel(r'$\hat{\beta}^H_t$',fontsize=13); ax.set_ylabel('Density',fontsize=13); ax.legend(fontsize=11)
plt.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR,'05_regime_density_KDE.png'),dpi=150,bbox_inches='tight')
plt.close(); print('  Saved.')

# ── Fig 6: Andrews sup-Wald ────────────────────────────────────────────────
print('Figure 6: Andrews sup-Wald...')
fig,axes=plt.subplots(1,2,figsize=(16,6))
for ax,bank in zip(axes,BANKS):
    aw=AW[bank]; ws=aw['ws']; idx=aw['idx']
    bank_col=C['BBVA'] if bank=='BBVA' else C['BCP']
    xs=[idx[i] for i in range(len(ws)) if not np.isnan(ws[i])]
    ys=[ws[i]  for i in range(len(ws)) if not np.isnan(ws[i])]
    ax.plot(xs,ys,color=bank_col,lw=1.2)
    ax.axhline(6.02, color='gold',  ls='--',lw=1.0,label='CV 10%=6.02')
    ax.axhline(8.85, color='orange',ls='--',lw=1.2,label='CV 5%=8.85')
    ax.axhline(12.16,color='red',   ls='--',lw=1.4,label='CV 1%=12.16')
    if aw['bpd']: ax.axvline(aw['bpd'],color='navy',ls=':',lw=1.5,
                             label=f'Opt. break: {str(aw["bpd"].date())}')
    ax.set_ylabel('Wald F-statistic',fontsize=13); ax.legend(fontsize=11)
plt.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR,'06_andrews_supwald.png'),dpi=150,bbox_inches='tight')
plt.close(); print('  Saved.')

# ── Fig 7: State-dependent coefficients ───────────────────────────────────
print('Figure 7: State-dependent model coefficients...')
fig,ax=plt.subplots(figsize=(12,6))
labels=['beta_H_base\n(calm)','beta_H_crisis\n(additional)','Total\n(crisis)']
x=np.arange(len(labels)); w=0.35
for i,bank in enumerate(BANKS):
    res=SD[bank]; m=res['m']; cov=m.cov_params()
    se_b=float(m.bse[1]); se_c=float(m.bse[2]); cov12=float(cov[1,2])
    se_tot=float(np.sqrt(se_b**2+se_c**2+2*cov12))
    vals=[res['bh'],res['bh_cr'],res['total_cr']]
    errs=[se_b*1.96,se_c*1.96,se_tot*1.96]
    bank_col=C['BBVA'] if bank=='BBVA' else C['BCP']
    off=(i-0.5)*w
    ax.bar(x+off,vals,w,label=bank,color=bank_col,alpha=0.7)
    ax.errorbar(x+off,vals,errs,fmt='none',color='black',capsize=4,lw=1.5)
ax.axhline(0,color=C['zero'],ls='--',lw=0.8)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('Coefficient',fontsize=13); ax.legend(fontsize=11)
plt.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR,'07_statedependent_coefs.png'),dpi=150,bbox_inches='tight')
plt.close(); print('  Saved.')

# ── Fig 8: Four-level significance summary ────────────────────────────────
print('Figure 8: Four-level significance summary...')
fig,axes=plt.subplots(2,2,figsize=(20,14))
axf=axes.flatten()
for i,bank in enumerate(BANKS):
    bh=RR[bank][W_PRI]['bh']; ts=RR[bank][W_PRI]['ts'].dropna()
    se=RR[bank][W_PRI]['se']
    bank_col=C['BBVA'] if bank=='BBVA' else C['BCP']
    ax=axf[i*2]; shade(ax); idx=bh.dropna().index
    ax.plot(idx,bh.loc[idx],color=bank_col,lw=1.2)
    ax.fill_between(idx,(bh-1.96*se).loc[idx],(bh+1.96*se).loc[idx],alpha=0.12,color=bank_col)
    ax.axhline(0,color=C['zero'],ls='--',lw=0.8)
    s_=SR_res[bank][W_PRI]; w_=WR_res[bank][W_PRI]; mw_=MW_res[bank]['Credit/liquidity only']
    info=(f'Sign test: {100*s_["prop"]:.0f}% neg, p={s_["p"]:.4f}{sig_star(s_["p"])}\n'
          f'Wilcoxon: p={w_["p"]:.4f}{sig_star(w_["p"])}\n'
          f'Mann-Whitney: p={mw_["p"]:.4f}{sig_star(mw_["p"])}\n'
          f'Cliff delta={mw_["d"]:+.3f} ({mw_["eff"]})')
    ax.text(0.02,0.05,info,transform=ax.transAxes,fontsize=8,va='bottom',
            bbox=dict(fc='white',alpha=0.85,ec='grey',boxstyle='round'))
    ax.set_ylabel(r'$\hat{\beta}^H_t$',fontsize=13)
    ax=axf[i*2+1]; shade(ax)
    ax.plot(ts.index,ts.values,color=bank_col,lw=0.9)
    ax.fill_between(ts.index,ts.values,-1.96,where=(ts.values<-1.96),color=C['sig'],alpha=0.45)
    ax.axhline(-1.96,color='red',ls='--',lw=1.0,label='-1.96')
    ax.axhline(-1.645,color='orange',ls=':',lw=0.9,label='-1.645')
    ax.axhline(0,color=C['zero'],ls='--',lw=0.7)
    aw_=AW[bank]
    ax.set_ylabel('t-value',fontsize=13); ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR,'08_significance_summary.png'),dpi=150,bbox_inches='tight')
plt.close(); print('  Saved.')

# ── Fig 9: LSTM feature ────────────────────────────────────────────────────
print('Figure 9: LSTM feature...')
fig,axes=plt.subplots(2,1,figsize=(16,10))
for ax,bank in zip(axes,BANKS):
    bh=RR[bank][W_PRI]['bh']; reg=REG[bank]; nt,pt=THR[bank]
    bank_col=C['BBVA'] if bank=='BBVA' else C['BCP']
    idx=bh.dropna().index; shade(ax)
    for st,col in REG_COLORS.items():
        mask=reg.loc[idx]==st
        if mask.any(): ax.scatter(idx[mask],bh.loc[idx[mask]],c=col,s=8,alpha=0.7,zorder=4)
    ax.plot(idx,bh.loc[idx],color=bank_col,lw=0.7,alpha=0.4,zorder=3)
    ax.axhline(0,color=C['zero'],ls='--',lw=0.8)
    ax.axhline(nt,color=C['hvuln'],ls=':',lw=0.9,alpha=0.8,label=f'mu-1sd={nt:.4f}')
    ax.axhline(pt,color=C['anom'], ls=':',lw=0.9,alpha=0.8,label=f'mu+1sd={pt:.4f}')
    ax.set_ylabel('beta_H_t (regime state variable)'); ax.legend(fontsize=11,loc='lower left')
plt.tight_layout()
fig.savefig(os.path.join(PLOTS_DIR,'09_lstm_feature.png'),dpi=150,bbox_inches='tight')
plt.close(); print('  Saved.')

print(f'\nAll 9 figures saved to: {PLOTS_DIR}')


────────────────── Section 11, Generating Figures ──────────────────
Figure 1: Rolling sensitivity W=104...
  Saved.
Figure 2: Rolling sensitivity W=52...
  Saved.
Figure 3: Three-state regime timeline...
  Saved.
Figure 4: Rolling t-statistics...
  Saved.
Figure 5: Regime density KDE...
  Saved.
Figure 6: Andrews sup-Wald...
  Saved.
Figure 7: State-dependent model coefficients...
  Saved.
Figure 8: Four-level significance summary...
  Saved.
Figure 9: LSTM feature...
  Saved.

All 9 figures saved to: /Users/167yiliqi/Downloads/TEST - PER/OUTPUT_CLEAN_PIPELINE/PLOTS


## Section 12, Thesis Summary Tables (7 CSV files)
All tables are saved to `OUTPUT_CLEAN_PIPELINE/TABLES/` as UTF-8 CSV files.

| File | Content |
|--|--|
| TableA | Full-sample WLS results (Newey-West HAC) |
| TableB | Rolling estimation diagnostic |
| TableC | H1 distributional tests (Sign + Wilcoxon) |
| TableD | H2 regime tests (Mann-Whitney + Cliff's delta) |
| TableE | Andrews (1993) sup-Wald structural break test |
| TableF | State-dependent model (crisis interaction) |
| TableG | Three-state regime classification summary |


In [14]:
sep('Section 12, Thesis Tables')

# Table A: Full-sample WLS
rows_A=[]
for bank in BANKS:
    t=FS[bank]['table'].copy(); t.insert(0,'Bank',bank); rows_A.append(t)
pd.concat(rows_A).to_csv(os.path.join(TABS_DIR,'TableA_FullSampleWLS.csv'),index=False)
print('  TableA saved: Full-Sample WLS (Newey-West HAC)')

# Table B: Rolling diagnostic
rows_B=[]
for bank in BANKS:
    for W in WINDOWS:
        bh=RR[bank][W]['bh'].dropna(); ts=RR[bank][W]['ts'].dropna()
        rows_B.append({'Bank':bank,'W':W,'N_valid':len(bh),
            'Pct_neg':f'{100*(bh<0).mean():.1f}%',
            'Pct_t_lt_196':f'{100*(ts<-1.96).mean():.1f}%',
            'Pct_t_lt_1645':f'{100*(ts<-1.645).mean():.1f}%',
            'Mean_bH':round(bh.mean(),6),'Median_bH':round(bh.median(),6),'Std_bH':round(bh.std(),6)})
pd.DataFrame(rows_B).to_csv(os.path.join(TABS_DIR,'TableB_RollingDiagnostic.csv'),index=False)
print('  TableB saved: Rolling estimation diagnostic')

# Table C: H1 tests
rows_C=[]
for bank in BANKS:
    for W in WINDOWS:
        s=SR_res[bank][W]; w=WR_res[bank][W]
        rows_C.append({'Bank':bank,'W':W,
            'Sign_n_neg':s['nn'],'Sign_n':s['n'],'Sign_prop':round(s['prop'],4),
            'Sign_p':round(s['p'],4),'Sign_sig':sig_star(s['p']),
            'Wilcoxon_median':round(w['med'],6) if w['med'] else np.nan,
            'Wilcoxon_p':round(w['p'],4) if w['p'] else np.nan,
            'Wilcoxon_sig':sig_star(w['p']) if w['p'] else ''})
pd.DataFrame(rows_C).to_csv(os.path.join(TABS_DIR,'TableC_H1Tests.csv'),index=False)
print('  TableC saved: H1 distributional tests')

# Table D: H2 tests
rows_D=[]
for bank in BANKS:
    for lbl in ['Credit/liquidity only','All crises (incl. political)']:
        r=MW_res[bank][lbl]
        rows_D.append({'Bank':bank,'Episodes':lbl,'W':W_PRI,
            'n_calm':r['nc'],'n_crisis':r['nk'],'U':round(r['U'],0),
            'p':round(r['p'],4),'Sig':sig_star(r['p']),
            'Cliffs_delta':round(r['d'],3),'Effect':r['eff'],
            'Mean_calm':round(r['mc'],6),'Mean_crisis':round(r['mk'],6)})
pd.DataFrame(rows_D).to_csv(os.path.join(TABS_DIR,'TableD_H2RegimeTests.csv'),index=False)
print('  TableD saved: H2 regime tests')

# Table E: Andrews sup-Wald
rows_E=[]
for bank in BANKS:
    aw=AW[bank]
    rows_E.append({'Bank':bank,'SupWald':round(aw['sw'],3),'p_approx':aw['pap'],
        'Optimal_break':str(aw['bpd'].date()) if aw['bpd'] else 'N/A',
        'CV_10pct':6.02,'CV_5pct':8.85,'CV_1pct':12.16})
pd.DataFrame(rows_E).to_csv(os.path.join(TABS_DIR,'TableE_AndrewsSupWald.csv'),index=False)
print('  TableE saved: Andrews sup-Wald test')

# Table F: State-dependent model
rows_F=[]
for bank in BANKS:
    t=SD[bank]['table'].copy(); t.insert(0,'Bank',bank); rows_F.append(t)
pd.concat(rows_F).to_csv(os.path.join(TABS_DIR,'TableF_StateDependentModel.csv'),index=False)
print('  TableF saved: State-dependent model')

# Table G: Regime summary
rows_G=[]
for bank in BANKS:
    bh=RR[bank][W_PRI]['bh'].dropna(); reg=REG[bank].dropna(); nt,pt=THR[bank]
    for st in ['high_vuln','calm','anomalous']:
        mask=reg==st; b_=bh[mask]
        rows_G.append({'Bank':bank,'State':st,'N_weeks':int(mask.sum()),
            'Pct_time':f'{100*mask.mean():.1f}%',
            'Mean_bH':round(float(b_.mean()) if len(b_)>0 else np.nan,6),
            'Median_bH':round(float(b_.median()) if len(b_)>0 else np.nan,6),
            'neg_threshold':round(nt,6),'pos_threshold':round(pt,6)})
pd.DataFrame(rows_G).to_csv(os.path.join(TABS_DIR,'TableG_RegimeSummary.csv'),index=False)
print('  TableG saved: Three-state regime summary')


──────────────────── Section 12, Thesis Tables ────────────────────
  TableA saved: Full-Sample WLS (Newey-West HAC)
  TableB saved: Rolling estimation diagnostic
  TableC saved: H1 distributional tests
  TableD saved: H2 regime tests
  TableE saved: Andrews sup-Wald test
  TableF saved: State-dependent model
  TableG saved: Three-state regime summary


## Section, -- Final Results Summary


In [15]:
sep('PIPELINE COMPLETE, Key Findings')
print()
print(f'Figures  → {PLOTS_DIR}')
print(f'Tables   → {TABS_DIR}')
print(f'LSTM CSV → {os.path.join(OUT_DIR,"clri_lstm_final.csv")}')
print()
print('='*65)
print('THESIS EVIDENCE MAP')
print('='*65)
print('H1 (beta_H < 0, bank equity exposed to credit stress):')
for bank in BANKS:
    s_=SR_res[bank][W_PRI]; w_=WR_res[bank][W_PRI]; fs=FS[bank]
    print(f'  {bank}: full-sample beta_H={fs["bh"]:+.5f}{sig_star(fs["ph"])} (one-sided)  '
          f'Sign p={s_["p"]:.4f}{sig_star(s_["p"])}  Wilcoxon p={w_["p"]:.4f}{sig_star(w_["p"])}')
print()
print('H2 (time variation, stronger in crisis):')
for bank in BANKS:
    mw_=MW_res[bank]['Credit/liquidity only']; aw_=AW[bank]
    print(f'  {bank}: Cliff delta={mw_["d"]:+.3f}({mw_["eff"]}) p={mw_["p"]:.4f}{sig_star(mw_["p"])}  '
          f'Andrews sup-Wald={aw_["sw"]:.2f}(p≈{aw_["pap"]})')
print()
print('H3 (LSTM forecast value):')
print('  Run horse race using clri_lstm_final.csv')
print('  Compare LSTM-AL (no CETI) vs LSTM-AL (+ beta_H_W104)')
print('  Evaluate: Kupiec POF, Christoffersen CC, Diebold-Mariano')
print('='*65)


───────────────── PIPELINE COMPLETE, Key Findings ─────────────────

Figures  → /Users/167yiliqi/Downloads/TEST - PER/OUTPUT_CLEAN_PIPELINE/PLOTS
Tables   → /Users/167yiliqi/Downloads/TEST - PER/OUTPUT_CLEAN_PIPELINE/TABLES
LSTM CSV → /Users/167yiliqi/Downloads/TEST - PER/OUTPUT_CLEAN_PIPELINE/clri_lstm_final.csv

THESIS EVIDENCE MAP
H1 (beta_H < 0, bank equity exposed to credit stress):
  BBVA: full-sample beta_H=-0.00224* (one-sided)  Sign p=0.0000***  Wilcoxon p=0.0000***
  BCP: full-sample beta_H=-0.00235(ns) (one-sided)  Sign p=0.0003***  Wilcoxon p=0.0000***

H2 (time variation, stronger in crisis):
  BBVA: Cliff delta=-0.206(medium) p=0.0002***  Andrews sup-Wald=2.23(p≈>0.10)
  BCP: Cliff delta=-0.536(large) p=0.0000***  Andrews sup-Wald=2.85(p≈>0.10)

H3 (LSTM forecast value):
  Run horse race using clri_lstm_final.csv
  Compare LSTM-AL (no CETI) vs LSTM-AL (+ beta_H_W104)
  Evaluate: Kupiec POF, Christoffersen CC, Diebold-Mariano
